## Tools-access_context 访问上下文
工具在能够访问运行时信息（例如对话历史记录、用户数据和持久内存）时，其功能最为强大。
是 LangChain 最新工具风格，专门让 工具（tool）能读取对话历史、状态、用户信息。

### 用途

之前写的工具都是独立函数,但现在这个工具 能看见整个聊天的上下文！
给工具开了一个上帝视角，能看到：

-  聊天历史 messages
-   全局状态 state
-   用户信息
-   配置

### State 短期记忆 
整个对话的状态仓库
里面存着：messages 消息列表 user_preferences 用户偏好 你想存啥就存啥

#### 访问状态 State

In [3]:
from langchain.messages import HumanMessage
from langchain.tools import ToolRuntime, tool


# ----------------------
# 工具 1：获取用户最后一句话
# ----------------------
@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]
    # 从聊天记录里，倒着往上翻，找到最近一句用户说的话，返回给 AI。
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return str(message.content)

    return "No user messages found"


# ----------------------
# 工具 2：获取用户偏好
# ----------------------
@tool
def get_user_preference(pref_name: str, runtime: ToolRuntime) -> str:
    """Get a user preference value."""
    # 从全局状态里，拿出用户保存的偏好设置（比如名字、喜欢颜色、时区）
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")

ToolRuntime 是 Agent 内部上下文，是由系统自动注入的，不是手动维护的

In [ ]:
# ----------------------
# 初始化本地 Ollama 模型
# ----------------------
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

# 创建 Agent
agent = create_agent(model=model, tools=[get_last_user_message, get_user_preference])

# 访问模型
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢蓝色，我叫小明，我刚才说我喜欢什么？")]},
    config={
        "metadata": {
            "state": {
                "messages": [
                    HumanMessage(content="我喜欢蓝色，我叫小明，我刚才说我喜欢什么？")
                ],
                "user_preferences": {"name": "小明", "favorite_color": "blue"},
            }
        }
    },
)

for msg in response["messages"]:
    print(msg.content)

我喜欢蓝色，我叫小明，我刚才说我喜欢什么？

我喜欢蓝色，我叫小明，我刚才说我喜欢什么？
根据我们之前的对话记录，您说你喜欢蓝色。请问接下来有什么我可以帮助您的吗？如果您有任何问题或需要进一步的信息，请告诉我！


#### 更新状态 State
用于 Command 更新代理的状态。
Command = 给 **LangGraph** 引擎的“指令”

##### Command 常见能力

> 1. 更新状态（你现在用的） Command(update={...}) <=> state.update(...)
> 2. 控制流程 Command(goto="next_node")
> 3. 多种组合 Command(update={...},goto="next_step")
>

##### 示例完整链路（LangGraph 完整执行流程）

```plaintext
用户输入：
“我叫小红”

↓
LLM 判断：
需要调用 set_user_name

↓
执行 Tool：
return Command(update={"user_name": "小红"})

↓
LangGraph 引擎：
state["user_name"] = "小红"   ✅（关键！）

↓
后续节点 / Tool：
可以直接用这个新状态
```



##### 用于LangGraph

LangChain：LLM + Tools = 推理 👉 工具只是辅助回答

🧠 LangGraph：LLM + Tools + State + Flow = 应用系统

👉 Tool 可以：

- 改状态

- 控流程
- 驱动整个应用

##### 思想转变

- LLM = 决策器
- Tool = 执行器
- State = 数据库
- Command = 写操作
- Graph = 工作流

In [29]:
from langchain.messages import ToolMessage
from langgraph.types import Command

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


# ----------------------
# Tool 1：写入用户名（关键：Command）
# LangChain 新版本
# Tool return → 自动转 ToolMessage
# state 修改 → runtime.state 直接改
# ----------------------
@tool
def set_user_name(new_name: str, runtime: ToolRuntime):
    """Set the user's name"""
    print("✅ set_user_name 被调用")
    # 1️⃣ 返回结果（给 LLM）
    result = f"用户名已设置为 {new_name}"

    # 2️⃣ 更新状态（用 Command）
    runtime.state["user_name"] = new_name

    return result

# ----------------------
# Tool 2：读取用户名并打招呼
# ----------------------
@tool
def greet_user(runtime: ToolRuntime) -> str:
    """Greet the user by name."""
    print("✅ greet_user 被调用")
    return f"你好 {runtime.state.get('user_name', '陌生人')}"


model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

# 创建 Agent
agent = create_agent(model=model, tools=[set_user_name, greet_user])

state = {}

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="""
请使用工具完成以下任务：

1. 使用 set_user_name 保存名字 Webster
2. 使用 greet_user 返回问候

注意：
最终答案必须直接来自工具返回值。
""")
        ]
    },
    config={"metadata": {"state": state}},
)

print(response["messages"][-1].content)

✅ set_user_name 被调用
✅ greet_user 被调用
✅ greet_user 被调用
✅ greet_user 被调用
✅ greet_user 被调用
✅ greet_user 被调用
It seems that despite setting your name and attempting to greet you, the `greet_user` function is not functioning as expected. Given this situation, I'll proceed with a general greeting since we have already set your name to 'Webster'.

Hello Webster!


##### Command 最新写法
工具 return 字符串 → LLM 能看到
Command 只负责修改状态
新版写法必须分开写

In [31]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command  # 👈 重点！

# ----------------------
# 工具：用 Command 直接修改状态
# ----------------------
@tool
def set_name(name: str, runtime: ToolRuntime):
    """Set user name and update state."""
    print("🔧 工具执行：set_name")
    
    # ✅ 核心：返回 Command，update 里面直接改 state
    return Command(
        update={"user_name": name},  # 👈 直接修改全局 state！
    ), f"用户名已设置：{name}"

# ----------------------
# 工具：读取状态
# ----------------------
@tool
def get_name(runtime: ToolRuntime):
    """Get current user name."""
    print("🔧 工具执行：get_name")
    return f"当前名字：{runtime.state.get('user_name', '无')}"

# ----------------------
# 模型
# ----------------------
model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M",
    model_provider="ollama"
)

# ----------------------
# 创建 Agent（必须用 create_react_agent）
# ----------------------
agent = create_agent(model, [set_name, get_name])

# ----------------------
# 运行
# ----------------------
response = agent.invoke({
    "messages": [
        HumanMessage(content="把名字设置成 Webster，然后告诉我当前名字")
    ]
})

# ----------------------
# 输出最终结果
# ----------------------
print("\n最终回答：")
print(response["messages"][-1].content)

🔧 工具执行：set_name
🔧 工具执行：get_name

最终回答：
你的名字已经成功设置为 Webster，现在告诉你当前的名字是 Webster。如果你有任何其他问题或需要进一步的帮助，请告诉我！


### Context 语境

上下文提供在调用时传递的 不可变配置数据。

可用于***用户 ID、会话详细信息或在对话过程中*不应更改的应用程序特定设置**。

它允许你在不让模型直接感知敏感信息（如 User ID）的情况下，把这些信息传递给工具执行函数。

##### 好处

1. 安全性 (Security)： 模型只负责“决定调什么”，敏感的身份标识（User ID、Token）由后台逻辑自动补全，模型接触不到。
2. 解耦 (Decoupling)： 提示词逻辑（AI 怎么说话）与业务逻辑（怎么查数据库）完全分离。

In [6]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain.chat_models import init_chat_model

USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}


@dataclass
class UserContext:
    user_id: str


@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f"Account holder: {user['name']}\nType: {user['account_type']}\nBalance: ${user['balance']}"

    return "User not found"


model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)
agent = create_agent(
    model,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a financial assistant.",
)

print("传入一个存在的user_id")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    context=UserContext(user_id="user123"),
)

for message in result["messages"]:
    print(message.pretty_print())

print("传入一个不存在的user_id")

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    context=UserContext(user_id="user789"),
)

for message in result["messages"]:
    print(message.pretty_print())

传入一个存在的user_id
================================ Human Message =================================

What's my current balance?
None
================================== Ai Message ==================================
Tool Calls:
  get_account_info (0a8567c5-742a-4493-ba92-64784f1c8330)
 Call ID: 0a8567c5-742a-4493-ba92-64784f1c8330
  Args:
None
================================= Tool Message =================================
Name: get_account_info

Account holder: Alice Johnson
Type: Premium
Balance: $5000
None
================================== Ai Message ==================================

Your current balance is $5000.
None
传入一个不存在的user_id
================================ Human Message =================================

What's my current balance?
None
================================== Ai Message ==================================
Tool Calls:
  get_account_info (20c24340-affb-4225-80cc-7e31b2c2a4c3)
 Call ID: 20c24340-affb-4225-80cc-7e31b2c2a4c3
  Args:
None
================================

#### Context 执行流程分解
1. ##### 注入 (Injection)：

  当你调用 agent.invoke(..., context=UserContext(user_id="user123")) 时，你并没有把 user123 写在 Prompt 里。这个对象被保存在了 Agent 的运行生命周期中。

2. ##### 模型决策 (Model Decision)：

  模型收到问题 "What's my current balance?"。它看到自己有一个工具叫 get_account_info，于是决定调用它。

* 重点： 注意看你的 tool_calls 里的 args 是空的 {}。模型并不知道 user_id 是什么，它只是发出了调用请求。

3. ##### 运行时调度 (Runtime Dispatch)：

  LangChain 的 ToolRuntime 介入。它发现模型要调用 get_account_info，同时它发现这个工具需要 UserContext。

  它自动从 agent.invoke 传入的 context 参数中提取出 UserContext(user_id="user123")。

  它将这个上下文对象注入到工具函数的 runtime 参数中。

4. ##### 业务逻辑执行 (Execution)：

  在 get_account_info 函数内部，通过 runtime.context.user_id 拿到了 user123，去 USER_DATABASE 查询数据并返回。

#### Agent 异常行为分析（魅力所在）
在 Agent 的思考链条里，发生了以下转换：

- 工具层： get_account_info 返回了 "User not found"。这是一个逻辑上的“失败”或“空值”。

- 模型层 (Qwen2.5)： 它接收到了这个返回结果。作为一名“Financial Assistant”（你在 system_prompt 中定义的角色），它意识到如果直接说“用户未找到”显得太生硬，甚至有点不专业。

- 输出层： 它自动补全了礼貌用语：“It seems there was an issue... Could you please provide more details...”。

> ## 引申 Agent和Model的区别
> 简单来说：**Model 是“大脑”，而 Agent 是“带了说明书和工具箱的职场人”。**
>
> 可以通过下面这个表格和流程分析来彻底理清它们的关系：
>
> ---
>
> #### 1. 概念大对碰：Model vs. Agent
>
> | 特性 | Model (模型 / 大脑) | Agent (代理 / 职场人) |
> | :--- | :--- | :--- |
> | **本质** | 一个复杂的数学函数（预测下一个字）。 | 一个运行程序，它**使用**模型来决策。 |
> | **能力范围** | 只有训练数据里的知识。无法直接联网、查数据库、算复杂的数。 | 可以通过“工具”连接现实世界（查余额、发邮件）。 |
> | **输入输出** | 输入文字，输出文字。 | 输入目标，输出一系列动作并达成结果。 |
> | **上下文感知** | 对 `Context` 本身无感知，它只看到被拼进 Prompt 的文字。 | **管理 Context 的专家**。它知道什么时候该把 Context 传给哪个工具。 |
>
> ---
>
> #### 2. 它们在代码流程中是如何“接力”的？
>
> `context_schema` 是在 Agent 创建时定义的，而 `invoke` 是在运行时赋值。这就是 Agent 在中间做的“翻译”工作。
>
> 
>
> ##### 整个执行链路是这样的：
>
> 1.  **用户发起请求：** 你给 **Agent** 发送指令 `"我的余额是多少？"`。
> 2.  **Agent 翻译上下文：** * **Agent** 拿着你传入的 `UserContext(user_id="user123")`。
>     * 它并**不直接**把这个 ID 给 **Model**（因为 Model 拿了 ID 也没地方查，反而可能造成隐私泄露）。
> 3.  **Model 做决策：**
>     * **Agent** 把问题发给 **Model**，并告诉它：“你有这些工具（比如 `get_account_info`），请决定下一步干嘛”。
>     * **Model** 思考后回复：“我想调用 `get_account_info`”。
> 4.  **Agent 执行动作（关键点）：**
>     * **Agent** 拦截到 Model 的请求。
>     * **Agent** 发现这个工具需要 `Context`。
>     * **Agent** 自动把手中的 `user123` 塞进工具里，然后运行工具。
> 5.  **Agent 汇报结果：** * 工具返回了结果，**Agent** 再次把结果发给 **Model**。
>     * **Model** 总结成一句话，最后通过 **Agent** 呈现给你。
>
> ---
>
> #### 3. 总结
>
> * **Model** 是**逻辑引擎**：它只负责“推理”。在你的例子里，它负责判断“这个用户想查余额，我应该调用那个查余额的工具”。
> * **Agent** 是**控制中心**：它负责“协调”。它负责拿着 `Context`、维护 `Memory`、调用 `Tools`。
>
> **为什么需要 `context_schema`？**
> 因为它像是一份**契约**。你告诉 Agent：“我有这类数据（比如用户 ID），以后如果有工具需要它，你就自动传过去，不用再麻烦 Model 去问我要了。”
>
> **比喻：**
> * **Model** 是一位非常有智慧但瘫痪在床的**先知**。
> * **Agent** 是先知的**助理**。
> * **Context** 是助理手里拿着的**客户档案**。
> * 当先知（Model）说“查一下这个人的余额”时，助理（Agent）不需要问先知“那人是谁”，而是直接翻看手中的档案（Context），帮先知把活儿干了。

### Long-term memory (Store) 长期存储

1. BaseStore 提供持久存储，数据可在会话之间保留。

2. 与状态（短期记忆）不同，保存到存储中的数据在以后的会话中仍然可用。

3. 存储库使用命名空间/键模式来组织数据。

#### 代码示例

```python
user_info = store.get(("users",), user_id)
store.put(("users",), user_id, user_info)
```

##### 第一个参数 ("users",) 是什么？

它是一个 Tuple（元组）。在 LangChain 的 Store 中，这被称为 Namespace（命名空间）。

* 为什么要用元组？ 为了支持层级结构。你可以把它想象成文件夹路径。

  > ("users",) 相当于 /users/ 文件夹。
  >
  > ("users", "premium") 相当于 /users/premium/ 文件夹。

* 为什么要这么写？

  > 如果你的 Agent 既要存“用户信息”，又要存“商品信息”，你可以用命名空间把它们隔离开：
  >
  > 用户信息存放在 ("users",) 下；商品信息存放在 ("products",) 下。
  >
  > 这样即使 user_id 和 product_id 碰巧重名了，也不会互相覆盖。

##### 第二个参数 user_id 是什么？

这是在这个命名空间下的 具体 Key。

连起来理解：

```python
store.put(("users",), user_id, user_info)
```

在 create_agent 的底层逻辑中，如果你在调用工具时指定了命名空间，它实际上是在执行：

“在 users 这个大分类下，把键值为 abc123 的数据存为 user_info。”

##### 最新版代码为什么是 mget / mset？

这里的 m 代表 Multiple（多个）。

* 设计哲学： 

  AI 助手有时需要一次性查询多个信息（比如查询 3 个订单的状态）。为了避免多次往返数据库（尤其是当你以后改用 Redis 或云数据库时），LangChain 强制使用批量接口来优化性能。

参数格式： 

```bash
 mset：必须传 [(键, 值)]。
 mget：必须传 [键1, 键2]。
```


###### 命名空间与存储

为什么会有这种写法？

LangChain 的底层存储结构支持 **Namespace (命名空间)**。它的存储逻辑其实是这样的：

1. **分层管理：** 就像电脑硬盘。
   - `("users",)` 就像是一个名为 `users` 的文件夹。
   - 如果你写 `("users", "admins")`，就像是 `users/admins/` 这种嵌套文件夹。
2. **隔离数据：** 如果你开发一个大型 Agent，可能既要存用户信息，又要存订单信息。
   - 如果你把所有东西都堆在一起，万一用户 ID 叫 `123`，订单 ID 也叫 `123`，数据就乱套了。
   - 有了命名空间，你可以把它们分别锁在不同的“抽屉”里。

为什么如上代码没报错也能跑通？

因为在 `create_agent` 的内部实现中，如果你直接操作 `runtime.store` 并且没有显式地去处理复杂的命名空间，它默认就在一个“根目录”下进行 `mget` 和 `mset`。

对于目前的本地测试，直接用 `user_id` 作为 Key 是完全没问题的。

> ### `dir()` 是什么作用？
>
> ```python
> print(dir(runtime.store))
> ```
>
> 在 Python 中，`dir()` 是一个非常有用的**“侦探工具”**（内置函数）。
>
> - **它的功能：** 返回一个列表，列出对象的所有**属性**（变量）和**方法**（函数）。
> - **为什么要用它：** 当你不确定一个对象（比如 `runtime.store`）到底有哪些招式可用时，用 `dir()` 刷一下就全出来了。
> - **怎么看结果：**
>   - 忽略带双下划线的（如 `__init__`），那是 Python 内部使用的。
>   - 关注那些普通名字，比如你刚才找的 `mget`, `mset`, `yield_keys` 等。
>
> > **比喻：** `dir()` 就像是你在玩游戏时，对着一个宝箱点“查看详情”，它会列出这个宝箱里所有的装备清单。

In [ ]:
from typing import Any, cast
from langchain.tools import tool
from langchain_core.stores import BaseStore, InMemoryStore


@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    if runtime.store is None:
        return "Store not available"

    # 使用 cast 明确类型，或者直接调用，忽略 IDE 的虚假报错
    store = cast(BaseStore[str, Any], runtime.store)
    
    # 注意：mget 接收的是 key 的列表
    results = store.mget([user_id])
    
    # results 会是一个列表 [value1, value2...]
    user_info = results[0] if results and results[0] is not None else None
    return str(user_info) if user_info else "Unknown user"

@tool
def save_user_info(
    user_id: str, user_info: dict[str, Any], runtime: ToolRuntime
) -> str:
    """Save user info."""
    if runtime.store is None:
        return "Store not available"

    store = cast(BaseStore[str, Any], runtime.store)
    
    # mset 接收的是 [(key, value), ...] 的列表
    store.mset([(user_id, user_info)])
    return "Successfully saved user info."


model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)
store = InMemoryStore()
agent = create_agent(
    model,
    tools=[get_user_info, save_user_info],
    store=store,
)

# First session: save user info
agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev",
            }
        ]
    }
)
# Second session: get user info
agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Get user info for user with id 'abc123'"}
        ]
    }
)

import json

# 不要直接 print(result)
# 而是循环消息列表，把它们转换成字典
formatted_messages = []
for msg in result["messages"]:
    # .dict() 会把 Message 对象变成 Python 字典，后续方便JSON打印
    formatted_messages.append(msg.dict())

# 使用 json.dumps 打印成真正的、带缩进的字符串
print(json.dumps(formatted_messages, indent=2, ensure_ascii=False))

[
  {
    "content": "What's my current balance?",
    "additional_kwargs": {},
    "response_metadata": {},
    "type": "human",
    "name": null,
    "id": "b8ff82c1-ffb2-4b51-b386-0728dd24cae3"
  },
  {
    "content": "",
    "additional_kwargs": {},
    "response_metadata": {
      "model": "qwen2.5:7b-instruct-q4_K_M",
      "created_at": "2026-04-08T14:43:38.5601319Z",
      "done": true,
      "done_reason": "stop",
      "total_duration": 1455182200,
      "load_duration": 114823100,
      "prompt_eval_count": 139,
      "prompt_eval_duration": 77375300,
      "eval_count": 17,
      "eval_duration": 1231709000,
      "logprobs": null,
      "model_name": "qwen2.5:7b-instruct-q4_K_M",
      "model_provider": "ollama"
    },
    "type": "ai",
    "name": null,
    "id": "lc_run--019d6d8c-7e0f-78e2-95ab-140f06b131e8-0",
    "tool_calls": [
      {
        "name": "get_account_info",
        "args": {},
        "id": "20c24340-affb-4225-80cc-7e31b2c2a4c3",
        "type": "tool_ca

C:\Users\Administrator\AppData\Local\Temp\ipykernel_24668\1914738312.py:74: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  formatted_messages.append(msg.dict())


### Stream writer 流处理器
在工具执行过程中，实时传输工具的更新信息。这对于在长时间运行的操作期间向用户提供进度反馈非常有用。
用于runtime.stream_writer发出自定义更新。

In [8]:
from langchain.tools import tool, ToolRuntime

@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a given city."""
    writer = runtime.stream_writer

    # Stream custom updates as the tool executes
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")

    return f"It's always sunny in {city}!"

### Execution info 执行信息
通过以下方式从工具内部访问线程 ID、运行 ID 和重试状态runtime.execution_info。

In [9]:
from langchain.tools import tool, ToolRuntime

@tool
def log_execution_context(runtime: ToolRuntime) -> str:
    """Log execution identity information."""
    info = runtime.execution_info
    print(f"Thread: {info.thread_id}, Run: {info.run_id}")
    print(f"Attempt: {info.node_attempt}")
    return "done"

### Server info 服务器信息
当您的工具运行在 LangGraph Server 上时，可通过以下方式访问助手 ID、图 ID 和已认证用户runtime.server_info：

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_assistant_scoped_data(runtime: ToolRuntime) -> str:
    """Fetch data scoped to the current assistant."""
    server = runtime.server_info

    if server is not None:
        print(f"Assistant: {server.assistant_id}, Graph: {server.graph_id}")
        if server.user is not None:
            print(f"User: {server.user.identity}")
            
    return "done"